In [3]:
from pathlib import Path
import sys

import duckdb
import pandas as pd


# ============================================================
# PROJECT CONFIG
# ============================================================

current = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in [current, *current.parents]
    if (p / "src" / "data_pipeline" / "config.py").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT / "src" / "data_pipeline")
)

from config import (
    BEHAVIOR_FEATURES_FILE,
    USER_BASELINE_FILE,
    GROUND_TRUTH_PARQUET,
    DATA_QUALITY_REPORT,
)


# ============================================================
# HELPERS
# ============================================================

def db_path(path):
    """Return an absolute path compatible with DuckDB."""
    return Path(path).resolve().as_posix()


def add_check(
    checks,
    name,
    result,
    expected,
    passed,
    severity="ERROR",
):
    """Add one data-quality check to the final report."""
    checks.append(
        {
            "Check": name,
            "Result": result,
            "Expected": expected,
            "Status": "PASS" if passed else "FAIL",
            "Severity": severity,
        }
    )


# ============================================================
# INPUT CHECK
# ============================================================

required_files = {
    "Behavior features": BEHAVIOR_FEATURES_FILE,
    "User baseline": USER_BASELINE_FILE,
    "Ground truth": GROUND_TRUTH_PARQUET,
}

missing = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not Path(path).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required validation files:\n"
        + "\n".join(missing)
    )

print("✓ Required validation files found.")


# ============================================================
# PATHS
# ============================================================

behavior = db_path(
    BEHAVIOR_FEATURES_FILE
)

baseline = db_path(
    USER_BASELINE_FILE
)

ground_truth = db_path(
    GROUND_TRUTH_PARQUET
)


# ============================================================
# START VALIDATION
# ============================================================

con = duckdb.connect()

checks = []

print("\n" + "=" * 70)
print("FINAL DATA QUALITY VALIDATION")
print("=" * 70)


# ============================================================
# 1. DATASET SUMMARY
# ============================================================

summary = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user) AS users,
        MIN(day) AS first_day,
        MAX(day) AS last_day

    FROM read_parquet(
        '{behavior}'
    )
""").df().iloc[0]


print("\n[1/13] Dataset summary")

print(
    f"      Rows       : "
    f"{int(summary['rows']):,}"
)

print(
    f"      Users      : "
    f"{int(summary['users']):,}"
)

print(
    f"      Date range : "
    f"{summary['first_day']} "
    f"→ {summary['last_day']}"
)


add_check(
    checks,
    "Unique users",
    int(summary["users"]),
    "1000",
    int(summary["users"]) == 1000,
)

add_check(
    checks,
    "Dataset contains rows",
    int(summary["rows"]),
    "> 0",
    int(summary["rows"]) > 0,
)


# ============================================================
# 2. USER-DAY UNIQUENESS
# ============================================================

duplicates = con.execute(f"""
    SELECT COUNT(*)

    FROM (
        SELECT
            user,
            day,
            COUNT(*) AS n

        FROM read_parquet(
            '{behavior}'
        )

        GROUP BY
            user,
            day

        HAVING COUNT(*) > 1
    )
""").fetchone()[0]


add_check(
    checks,
    "Duplicate user-day records",
    duplicates,
    "0",
    duplicates == 0,
)

print("[2/13] User-Day uniqueness checked.")


# ============================================================
# 3. CRITICAL NULL CHECK
# ============================================================

critical_columns = [
    "user",
    "day",
    "email_count",
    "external_email_events",
    "usb_connect_count",
    "files_copied_to_usb",
    "http_request_count",
    "unique_domains",
    "logon_count",
    "baseline_workdays",
    "usb_files_p95",
    "external_email_p95",
    "http_requests_p95",
]


for column in critical_columns:

    null_count = con.execute(f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{behavior}'
        )

        WHERE "{column}" IS NULL
    """).fetchone()[0]

    add_check(
        checks,
        f"Null values: {column}",
        null_count,
        "0",
        null_count == 0,
    )


print("[3/13] Critical nulls checked.")


# ============================================================
# 4. NEGATIVE VALUE CHECK
# ============================================================

non_negative_columns = [
    "email_count",
    "external_email_events",
    "total_email_bytes",
    "attachment_count",
    "usb_connect_count",
    "usb_disconnect_count",
    "files_copied_to_usb",
    "unique_file_extensions",
    "http_request_count",
    "unique_domains",
    "logon_count",
    "early_logon_minutes",
    "late_logoff_minutes",
    "daily_abnormal_feature_count",
    "external_emails_7d",
    "usb_files_7d",
    "http_requests_7d",
    "abnormal_days_7d",
    "external_emails_14d",
    "usb_files_14d",
    "http_requests_14d",
    "abnormal_days_14d",
    "usb_files_30d",
    "external_emails_30d",
    "abnormal_days_30d",
]


for column in non_negative_columns:

    negative_count = con.execute(f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{behavior}'
        )

        WHERE "{column}" < 0
    """).fetchone()[0]

    add_check(
        checks,
        f"Negative values: {column}",
        negative_count,
        "0",
        negative_count == 0,
    )


print("[4/13] Non-negative features checked.")


# ============================================================
# 5. LOGICAL CONSISTENCY
# ============================================================

invalid_external_email = con.execute(f"""
    SELECT COUNT(*)

    FROM read_parquet(
        '{behavior}'
    )

    WHERE external_email_events > email_count
""").fetchone()[0]


add_check(
    checks,
    "External email <= total email",
    invalid_external_email,
    "0 invalid rows",
    invalid_external_email == 0,
)


invalid_extensions = con.execute(f"""
    SELECT COUNT(*)

    FROM read_parquet(
        '{behavior}'
    )

    WHERE unique_file_extensions
          > files_copied_to_usb
""").fetchone()[0]


add_check(
    checks,
    "Unique file extensions <= copied files",
    invalid_extensions,
    "0 invalid rows",
    invalid_extensions == 0,
)


invalid_weekend = con.execute(f"""
    SELECT COUNT(*)

    FROM read_parquet(
        '{behavior}'
    )

    WHERE is_weekend NOT IN (0, 1)
       OR is_weekend IS NULL
""").fetchone()[0]


add_check(
    checks,
    "Weekend flag validity",
    invalid_weekend,
    "0 invalid rows",
    invalid_weekend == 0,
)


print("[5/13] Logical consistency checked.")


# ============================================================
# 6. BINARY FLAG VALIDATION
# ============================================================

flag_columns = [
    "email_above_p95",
    "external_email_above_p95",
    "usb_connect_above_p95",
    "usb_files_above_p95",
    "http_above_p95",
    "domains_above_p95",
    "abnormal_day",
]


for column in flag_columns:

    invalid = con.execute(f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{behavior}'
        )

        WHERE "{column}" NOT IN (0, 1)
           OR "{column}" IS NULL
    """).fetchone()[0]

    add_check(
        checks,
        f"Binary flag validity: {column}",
        invalid,
        "0 invalid rows",
        invalid == 0,
    )


print("[6/13] Binary flags checked.")


# ============================================================
# 7. BASELINE QUALITY
# ============================================================

baseline_summary = con.execute(f"""
    SELECT
        COUNT(*) AS users,

        MIN(
            baseline_workdays
        ) AS min_days,

        MEDIAN(
            baseline_workdays
        ) AS median_days,

        MAX(
            baseline_workdays
        ) AS max_days

    FROM read_parquet(
        '{baseline}'
    )
""").df().iloc[0]


add_check(
    checks,
    "Baseline users",
    int(baseline_summary["users"]),
    "1000",
    int(baseline_summary["users"]) == 1000,
)

add_check(
    checks,
    "Minimum baseline history",
    int(baseline_summary["min_days"]),
    ">= 20 workdays",
    int(baseline_summary["min_days"]) >= 20,
)


print("[7/13] Baseline quality checked.")


# ============================================================
# 8. GROUND TRUTH VALIDATION
# ============================================================

gt = con.execute(f"""
    SELECT
        COUNT(*) AS incidents,

        COUNT(
            DISTINCT user
        ) AS malicious_users,

        SUM(
            CASE
                WHEN exfiltration_relevant = 1
                THEN 1
                ELSE 0
            END
        ) AS exfiltration_incidents

    FROM read_parquet(
        '{ground_truth}'
    )
""").df().iloc[0]


add_check(
    checks,
    "Ground-truth incidents",
    int(gt["incidents"]),
    "70",
    int(gt["incidents"]) == 70,
)

add_check(
    checks,
    "Unique malicious users",
    int(gt["malicious_users"]),
    "70",
    int(gt["malicious_users"]) == 70,
)

add_check(
    checks,
    "Exfiltration-relevant incidents",
    int(gt["exfiltration_incidents"]),
    "60",
    int(gt["exfiltration_incidents"]) == 60,
)


print("[8/13] Ground truth checked.")


# ============================================================
# 9. GROUND-TRUTH USERS EXIST IN BEHAVIOR DATA
# ============================================================

missing_attack_users = con.execute(f"""
    SELECT COUNT(*)

    FROM (
        SELECT DISTINCT
            g.user

        FROM read_parquet(
            '{ground_truth}'
        ) g

        LEFT JOIN (

            SELECT DISTINCT
                user

            FROM read_parquet(
                '{behavior}'
            )

        ) b

        USING(user)

        WHERE b.user IS NULL
    )
""").fetchone()[0]


add_check(
    checks,
    "Ground-truth users present in behavioral dataset",
    missing_attack_users,
    "0 missing users",
    missing_attack_users == 0,
)


print("[9/13] Ground-truth users checked.")


# ============================================================
# 10. ATTACK DATES INSIDE DATA RANGE
# ============================================================

invalid_attack_dates = con.execute(f"""
    WITH bounds AS (

        SELECT
            MIN(day) AS first_day,
            MAX(day) AS last_day

        FROM read_parquet(
            '{behavior}'
        )
    )

    SELECT COUNT(*)

    FROM read_parquet(
        '{ground_truth}'
    ),
    bounds

    WHERE
        CAST(
            attack_start AS DATE
        ) < first_day

        OR

        CAST(
            attack_end AS DATE
        ) > last_day
""").fetchone()[0]


add_check(
    checks,
    "Attack dates within dataset period",
    invalid_attack_dates,
    "0 outside range",
    invalid_attack_dates == 0,
)


print("[10/13] Attack-date range checked.")


# ============================================================
# 11. EXTERNAL EMAIL SANITY CHECK
# ============================================================

email_check = con.execute(f"""
    SELECT
        SUM(
            email_count
        ) AS total_emails,

        SUM(
            external_email_events
        ) AS external_emails,

        ROUND(
            100.0
            * SUM(
                external_email_events
            )
            / NULLIF(
                SUM(email_count),
                0
            ),
            2
        ) AS percentage

    FROM read_parquet(
        '{behavior}'
    )
""").df().iloc[0]


email_percentage = float(
    email_check["percentage"]
)


add_check(
    checks,
    "External email percentage",
    email_percentage,
    "Between 0% and 100%",
    0 < email_percentage < 100,
)


print(
    "[11/13] Email sanity checked "
    f"({email_percentage:.2f}%)."
)


# ============================================================
# 12. KNOWN CERT CASE
# ============================================================

known_case = con.execute(f"""
    SELECT
        files_copied_to_usb,
        usb_files_p95,

        ROUND(
            usb_files_p95_ratio,
            2
        ) AS usb_ratio,

        usb_files_above_p95

    FROM read_parquet(
        '{behavior}'
    )

    WHERE user = 'AAF0535'
      AND day = DATE '2010-08-10'
""").df()


known_case_ok = (
    len(known_case) == 1

    and int(
        known_case.iloc[0][
            "files_copied_to_usb"
        ]
    ) == 11

    and float(
        known_case.iloc[0][
            "usb_files_p95"
        ]
    ) == 5.0

    and int(
        known_case.iloc[0][
            "usb_files_above_p95"
        ]
    ) == 1
)


add_check(
    checks,
    "Known malicious behavior sanity check",
    "AAF0535 / 2010-08-10",
    "11 files, P95=5, above_p95=1",
    known_case_ok,
)


print("[12/13] Known CERT case checked.")


# ============================================================
# 13. BASELINE CONSISTENCY CHECK
# ============================================================

baseline_consistency = con.execute(f"""
    WITH cutoff AS (

        SELECT
            CAST(
                MIN(attack_start)
                AS DATE
            ) AS baseline_cutoff

        FROM read_parquet(
            '{ground_truth}'
        )
    ),

    calculated AS (

        SELECT
            MEDIAN(
                external_email_events
            ) AS calculated_external_median,

            QUANTILE_CONT(
                external_email_events,
                0.95
            ) AS calculated_external_p95,

            MEDIAN(
                files_copied_to_usb
            ) AS calculated_usb_median,

            QUANTILE_CONT(
                files_copied_to_usb,
                0.95
            ) AS calculated_usb_p95

        FROM read_parquet(
            '{behavior}'
        ),
        cutoff

        WHERE user = 'AAF0535'
          AND day < baseline_cutoff
          AND is_weekend = 0
          AND logon_count > 0
    ),

    saved AS (

        SELECT
            external_email_median,
            external_email_p95,
            usb_files_median,
            usb_files_p95

        FROM read_parquet(
            '{baseline}'
        )

        WHERE user = 'AAF0535'
    )

    SELECT *
    FROM calculated,
         saved
""").df().iloc[0]


baseline_matches = (

    float(
        baseline_consistency[
            "calculated_external_median"
        ]
    )
    ==
    float(
        baseline_consistency[
            "external_email_median"
        ]
    )

    and

    abs(
        float(
            baseline_consistency[
                "calculated_external_p95"
            ]
        )
        -
        float(
            baseline_consistency[
                "external_email_p95"
            ]
        )
    ) < 0.0001

    and

    float(
        baseline_consistency[
            "calculated_usb_median"
        ]
    )
    ==
    float(
        baseline_consistency[
            "usb_files_median"
        ]
    )

    and

    abs(
        float(
            baseline_consistency[
                "calculated_usb_p95"
            ]
        )
        -
        float(
            baseline_consistency[
                "usb_files_p95"
            ]
        )
    ) < 0.0001
)


add_check(
    checks,
    "Baseline consistent with current behavioral data",
    "AAF0535 baseline values",
    "Calculated values = saved baseline values",
    baseline_matches,
)


print("[13/13] Baseline consistency checked.")


# ============================================================
# BUILD FINAL REPORT
# ============================================================

report = pd.DataFrame(
    checks
)

failed = report[
    report["Status"] == "FAIL"
]

passed = int(
    (
        report["Status"] == "PASS"
    ).sum()
)


# ============================================================
# VALIDATE EXPECTED NUMBER OF CHECKS
# ============================================================

assert len(report) == 61, (
    f"Expected 61 validation checks, "
    f"but found {len(report)}."
)


# ============================================================
# SAVE OFFICIAL REPORT
# ============================================================

DATA_QUALITY_REPORT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

report.to_csv(
    DATA_QUALITY_REPORT,
    index=False,
)


# ============================================================
# PREPARE FULL NOTEBOOK TABLE
# ============================================================

display_report = report.copy()

display_report.insert(
    0,
    "No.",
    range(
        1,
        len(display_report) + 1
    ),
)


# ============================================================
# DISPLAY ALL 61 CHECKS
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATA QUALITY REPORT — ALL 61 CHECKS")
print("=" * 70)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(
        display_report.style
        .hide(axis="index")
    )


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(
    f"Total checks : "
    f"{len(report)}"
)

print(
    f"Passed       : "
    f"{passed}"
)

print(
    f"Failed       : "
    f"{len(failed)}"
)


if failed.empty:

    print(
        "\n✅ DATA QUALITY STATUS: PASS"
    )

    print(
        "Dataset is ready for handoff."
    )

else:

    print(
        "\n❌ DATA QUALITY STATUS: "
        "REVIEW REQUIRED"
    )

    print("\nFailed checks:")

    failed_display = (
        display_report[
            display_report["Status"] == "FAIL"
        ]
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
    ):
        display(
            failed_display.style
            .hide(axis="index")
        )


print(
    f"\nReport saved to:\n"
    f"{DATA_QUALITY_REPORT}"
)


# ============================================================
# CLOSE CONNECTION
# ============================================================

con.close()


# ============================================================
# FAIL NOTEBOOK IF ANY CHECK FAILED
# ============================================================

if not failed.empty:

    raise RuntimeError(
        f"Data quality validation failed: "
        f"{len(failed)} check(s) failed."
    )

✓ Required validation files found.

FINAL DATA QUALITY VALIDATION

[1/13] Dataset summary
      Rows       : 464,687
      Users      : 1,000
      Date range : 2010-01-02 00:00:00 → 2011-05-17 00:00:00
[2/13] User-Day uniqueness checked.
[3/13] Critical nulls checked.
[4/13] Non-negative features checked.
[5/13] Logical consistency checked.
[6/13] Binary flags checked.
[7/13] Baseline quality checked.
[8/13] Ground truth checked.
[9/13] Ground-truth users checked.
[10/13] Attack-date range checked.
[11/13] Email sanity checked (48.06%).
[12/13] Known CERT case checked.
[13/13] Baseline consistency checked.

FINAL DATA QUALITY REPORT — ALL 61 CHECKS


No.,Check,Result,Expected,Status,Severity
1,Unique users,1000,1000,PASS,ERROR
2,Dataset contains rows,464687,> 0,PASS,ERROR
3,Duplicate user-day records,0,0,PASS,ERROR
4,Null values: user,0,0,PASS,ERROR
5,Null values: day,0,0,PASS,ERROR
6,Null values: email_count,0,0,PASS,ERROR
7,Null values: external_email_events,0,0,PASS,ERROR
8,Null values: usb_connect_count,0,0,PASS,ERROR
9,Null values: files_copied_to_usb,0,0,PASS,ERROR
10,Null values: http_request_count,0,0,PASS,ERROR



FINAL RESULT
Total checks : 61
Passed       : 61
Failed       : 0

✅ DATA QUALITY STATUS: PASS
Dataset is ready for handoff.

Report saved to:
C:\Users\HP\Documents\FYP\Data_engineering\docs\data_quality_report.csv
